# Model Experiment


## 1. Imports and Setup


In [1]:
import torch
import torch.nn as nn

from model import ResNet34
from torch.optim import AdamW
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import Subset, DataLoader

import tqdm

import subprocess
from pathlib import Path

In [2]:
PROJECT_ROOT = Path(
    subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"],
        text=True
    ).strip()
)

print(PROJECT_ROOT)

/mnt/c/Users/ramom/Desktop/computer-vision-architectures


In [3]:
DATA_DIR = PROJECT_ROOT / 'data'

if not DATA_DIR.exists():
    Path.mkdir(DATA_DIR)

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {device}')

Using: cuda


## 2. Configuration


In [5]:
NUM_CLASSES = 2
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

## 3. Model


In [6]:
model = ResNet34(NUM_CLASSES).to(device)

## 4. Smoke Test


In [7]:
x = torch.rand(4, 3, 224, 224).to(device)

with torch.no_grad():
    y = model(x)

print(f'Input shape: {x.shape}')
print(f'Output shape: {y.shape}')

assert y.shape == (x.shape[0], NUM_CLASSES)

Input shape: torch.Size([4, 3, 224, 224])
Output shape: torch.Size([4, 2])


## 5. Dataset & DataLoaders


### 5.1 Data Augmentation Transforms

In [8]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

### 5.2 Take a Subset of Data

In [9]:
base_dataset = ImageFolder(DATA_DIR / 'cats-vs-dogs', transform=transform)

n = len(base_dataset)

generator = torch.Generator().manual_seed(42)
indices = torch.randperm(n, generator=generator)

end = int(0.004 * n)
indices = indices[:end]
dataset = Subset(base_dataset, indices)
data_loader = DataLoader(dataset, batch_size=8, shuffle=False)
print(len(dataset))

100


## 6. Loss & Optimizer


In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE, 
    weight_decay=WEIGHT_DECAY
)

## 7. Training


### 7.1 Helping Functions

In [11]:
def train_batch(
    images: torch.Tensor,
    labels: torch.Tensor,
    model: ResNet34,
    criterion: nn.CrossEntropyLoss,
    optimizer: torch.optim.SGD
):
    model.train()
    logits = model(images)
    loss = criterion(logits, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

@torch.no_grad()
def test_batch(
    images: torch.Tensor,
    labels: torch.Tensor,
    model: ResNet34,
    criterion: nn.CrossEntropyLoss,
):
    model.eval()
    logits = model(images)
    loss = criterion(logits, labels)

    return loss.item()

### 7.2 Training Loop Overfit Test

In [12]:
for epoch in range(1, 51):
    total_loss = 0.0

    for images, labels in data_loader:
        images = images.to(device)
        labels = labels.to(device).long()

        loss = train_batch(
            images,
            labels,
            model,
            criterion,
            optimizer
        )

        total_loss += loss

    avg_loss = total_loss / len(data_loader)

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch}: "
            f"Loss={avg_loss:.4f}"
        )

Epoch 1: Loss=0.7171
Epoch 5: Loss=0.3504
Epoch 10: Loss=0.1542
Epoch 15: Loss=0.0580
Epoch 20: Loss=0.0225
Epoch 25: Loss=0.0131
Epoch 30: Loss=0.0095
Epoch 35: Loss=0.0073
Epoch 40: Loss=0.0059
Epoch 45: Loss=0.0048
Epoch 50: Loss=0.0040
